# Explicación Detallada de la Práctica 4
### Generadores Congruenciales de Periodo Completo y el Teorema de Hull-Dobell

Este cuaderno explica y ejecuta los algoritmos contenidos en `Practica4.py`.
El objetivo principal es entender cómo configurar los parámetros del Generador Congruencial Lineal (LCG) para garantizar un **periodo completo** (es decir, que genere todos los números posibles dentro del módulo $M$ antes de repetir cualquier valor).


## El Teorema de Hull-Dobell

Para que un Generador Congruencial Mixto ($X_i = (a X_{i-1} + b) \pmod M$) alcance su **periodo máximo** o **periodo completo** (que es igual a $M$), sus parámetros deben satisfacer tres condiciones matemáticas estrictas:

1. **Coprimalidad**: El incremento $b$ y el módulo $M$ deben ser primos entre sí (es decir, su máximo común divisor debe ser 1: $\text{mcd}(b, M) = 1$).
2. **Divisibilidad de Factores Primos**: El valor $a - 1$ debe ser divisible por todos los factores primos de $M$.
3. **Condición del 4**: Si el módulo $M$ es múltiplo de 4, entonces $a - 1$ también debe ser múltiplo de 4.

### Aplicación a nuestro caso ($M = 4096$):
* El módulo $M = 4096 = 2^{12}$ es una potencia de 2. Su único factor primo es **2**.
* Para cumplir la **Condición 1**: Elegimos un incremento $b$ impar (por ejemplo, $b = 11$). Al ser impar y $M$ una potencia de 2, son coprimos: $\text{mcd}(11, 4096) = 1$.
* Para cumplir las **Condiciones 2 y 3**: El valor $a - 1$ debe ser divisible por 2 y por 4 (ya que 4096 es múltiplo de 4). Si elegimos $a = 21$, entonces $a - 1 = 20$, que es divisible tanto por 2 como por 4.
* **Conclusión**: Con $a=21, b=11, M=4096$, el teorema garantiza que el generador tendrá un periodo completo de **4096** números únicos.


In [1]:
def generador_periodo_completo(semilla, a, b, M):
    secuencia = []
    registro_apariciones = {}
    
    X = semilla % M
    ciclo_completo_alcanzado = False
    
    print(f"Configuración del espacio modular:")
    print(f"Semilla operativa inicial: {X} | Multiplicador (a): {a} | Incremento (b): {b} | Módulo (M): {M}")
    print("-" * 85)

    # Bucle hasta M + 1 para capturar la iteración exacta de la repetición
    for i in range(1, M + 2):
        X = (a * X + b) % M
        
        if X in registro_apariciones:
            p_primera = registro_apariciones[X]
            periodo_real = i - p_primera
            print(f"-> ¡Ciclo interceptado en la iteración N° {i}! (Valor repetido: {X})")
            print(f"   Primera aparición: Iteración {p_primera}")
            print(f"   Periodo real alcanzado: {periodo_real} números pseudoaleatorios.")
            
            if periodo_real == M:
                print("   🏆 ¡ÉXITO! Se ha demostrado matemáticamente el PERIODO COMPLETO.")
                ciclo_completo_alcanzado = True
            else:
                print("   ⚠️ ADVERTENCIA: El periodo es parcial. Las constantes no son óptimas.")
            break
            
        registro_apariciones[X] = i
        secuencia.append(X)
        
    return secuencia if ciclo_completo_alcanzado else []

def mostrar_muestras_justificadas(secuencia, total_a_mostrar=50):
    # Muestra los primeros y últimos elementos para no saturar la pantalla
    print(f"\nVisualización del periodo generado (Muestra de los primeros y últimos {total_a_mostrar} números):")
    print("=" * 85)
    
    print(f"--- PRIMEROS {total_a_mostrar} NÚMEROS ---")
    bloque_inicio = secuencia[:total_a_mostrar]
    for i in range(0, len(bloque_inicio), 10):
        linea = "".join(f"{num:<8}" for num in bloque_inicio[i:i+10])
        print(f"   {linea}")
        
    print("\n   ... [Se omiten los números intermedios del periodo completo] ...\n")
    
    print(f"--- ÚLTIMOS {total_a_mostrar} NÚMEROS ---")
    bloque_fin = secuencia[-total_a_mostrar:]
    for i in range(0, len(bloque_fin), 10):
        linea = "".join(f"{num:<8}" for num in bloque_fin[i:i+10])
        print(f"   {linea}")
    print("=" * 85)


### Ejecución con Parámetros Óptimos (Periodo Completo)
Ejecutaremos el algoritmo utilizando los valores sugeridos por defecto que cumplen el Teorema de Hull-Dobell para $M=4096$.
Comprobaremos que efectivamente se alcanza el periodo completo.


In [2]:
semilla_input = 7326
a_input = 21
b_input = 11
M_input = 4096

print("GENERADOR DE NÚMEROS PSEUDOALEATORIOS - PERIODO MÁXIMO")
print("=" * 85)

# Ejecución del algoritmo
numeros_generados = generador_periodo_completo(semilla_input, a_input, b_input, M_input)

if numeros_generados:
    mostrar_muestras_justificadas(numeros_generados, total_a_mostrar=50)


GENERADOR DE NÚMEROS PSEUDOALEATORIOS - PERIODO MÁXIMO
Configuración del espacio modular:
Semilla operativa inicial: 3230 | Multiplicador (a): 21 | Incremento (b): 11 | Módulo (M): 4096
-------------------------------------------------------------------------------------
-> ¡Ciclo interceptado en la iteración N° 4097! (Valor repetido: 2305)
   Primera aparición: Iteración 1
   Periodo real alcanzado: 4096 números pseudoaleatorios.
   🏆 ¡ÉXITO! Se ha demostrado matemáticamente el PERIODO COMPLETO.

Visualización del periodo generado (Muestra de los primeros y últimos 50 números):
--- PRIMEROS 50 NÚMEROS ---
   2305    3360    939     3346    645     1268    2063    2374    713     2696    
   3379    1338    3533    476     1815    1262    1937    3824    2491    3170    
   1045    1476    2335    3990    1881    2648    2371    650     1373    172     
   3623    2366    545     3264    3019    1970    421     660     1583    486     
   2025    1576    339     3034    2285    2940   

## Experimento: Violación del Teorema de Hull-Dobell (Periodo Parcial)

Para comprender el valor del teorema de Hull-Dobell, realicemos dos experimentos violando sus condiciones y observemos cómo el periodo se reduce drásticamente.

### Caso A: El incremento $b$ no es coprimo con $M$
Elegiremos $b = 10$ (par) y $M = 4096$ (par). Ambos comparten el factor primo 2, por lo que $\text{mcd}(10, 4096) = 2 \ne 1$.

### Caso B: El multiplicador $a$ no cumple las condiciones de divisibilidad
Elegiremos $a = 20$, por lo que $a - 1 = 19$. El número 19 no es divisible por 2 (factor primo de $M$), violando las condiciones 2 y 3.


In [3]:
# Caso A: b no es coprimo con M (ambos son pares)
print("EXPERIMENTO A: Incremento 'b' no coprimo con M (b=10, M=4096)")
print("Esperamos que el periodo sea muy corto porque viola la condición 1 (coprimalidad).")
generador_periodo_completo(semilla=7326, a=21, b=10, M=4096)

print("\n" + "="*85 + "\n")

# Caso B: a-1 no es divisible por los factores de M
print("EXPERIMENTO B: Multiplicador 'a' no cumple condiciones (a=20, M=4096)")
print("Esperamos un periodo parcial ya que a-1=19 no es divisible por 2 ni por 4.")
generador_periodo_completo(semilla=7326, a=20, b=11, M=4096)


EXPERIMENTO A: Incremento 'b' no coprimo con M (b=10, M=4096)
Esperamos que el periodo sea muy corto porque viola la condición 1 (coprimalidad).
Configuración del espacio modular:
Semilla operativa inicial: 3230 | Multiplicador (a): 21 | Incremento (b): 10 | Módulo (M): 4096
-------------------------------------------------------------------------------------
-> ¡Ciclo interceptado en la iteración N° 2049! (Valor repetido: 2304)
   Primera aparición: Iteración 1
   Periodo real alcanzado: 2048 números pseudoaleatorios.
   ⚠️ ADVERTENCIA: El periodo es parcial. Las constantes no son óptimas.


EXPERIMENTO B: Multiplicador 'a' no cumple condiciones (a=20, M=4096)
Esperamos un periodo parcial ya que a-1=19 no es divisible por 2 ni por 4.
Configuración del espacio modular:
Semilla operativa inicial: 3230 | Multiplicador (a): 20 | Incremento (b): 11 | Módulo (M): 4096
-------------------------------------------------------------------------------------
-> ¡Ciclo interceptado en la iteración

[]